A notebook to note down how I imported existing vocabularies, using epos-voc-env conda environment

# QUDT vocabulary for units and unit types

In [1]:
import pandas as pd
from rdflib import Graph

# 1. Versione rigida e immutabile
VERSION = "3.1.10" 
URL_VERSIONATO = f"http://qudt.org/{VERSION}/vocab/unit"

print(f"Download della versione immutabile QUDT Units {VERSION}...")
g = Graph()
g.parse(URL_VERSIONATO, format="turtle")

# 2. Query SPARQL con RAGGRUPPAMENTO (GROUP BY) per garantire l'unicità dell'URI
query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX qudt: <http://qudt.org/schema/qudt/>

SELECT ?unitURI 
       (SAMPLE(?label) AS ?unitLabel) 
       (SAMPLE(?symbol) AS ?unitSymbol) 
       (SAMPLE(?conversionMultiplier) AS ?multiplier)
       (GROUP_CONCAT(DISTINCT ?quantityKind; separator="|") AS ?quantityKinds)
WHERE {
  ?unitURI rdf:type qudt:Unit .
  
  # Estraiamo la label e filtriamo subito per l'inglese
  ?unitURI rdfs:label ?label .
  FILTER(lang(?label) = "en")
  
  OPTIONAL { ?unitURI qudt:symbol ?symbol . }
  OPTIONAL { ?unitURI qudt:conversionMultiplier ?conversionMultiplier . }
  OPTIONAL { ?unitURI qudt:hasQuantityKind ?quantityKind . }
}
GROUP BY ?unitURI
"""

print("Esecuzione del raggruppamento semantico locale...")
results = g.query(query)

# 3. Creazione del dizionario tabulare (una riga per ogni URI unico)
rows = []
for row in results:
    rows.append({
        "Unit_URI": str(row.unitURI),
        "Label": str(row.unitLabel) if row.unitLabel else "",
        "Symbol": str(row.unitSymbol) if row.unitSymbol else "",
        "Conversion_Multiplier": str(row.multiplier) if row.multiplier else "",
        # Questa colonna conterrà gli URI delle grandezze separati da "|" (es. URI_1|URI_2)
        "Quantity_Kind_URIs": str(row.quantityKinds) if row.quantityKinds else "",
        "Vocab_Version": f"QUDT-v{VERSION}"
    })

# 4. Generazione del CSV finale
df = pd.DataFrame(rows)
output_file = f"qudt_units_unique_v{VERSION}.csv"
df.to_csv(output_file, index=False, encoding="utf-8")

print(f"Completato! Generato il file '{output_file}' con {len(df)} URI univoci.")

Download della versione immutabile QUDT Units 3.1.10...
Esecuzione del raggruppamento semantico locale...
Completato! Generato il file 'qudt_units_unique_v3.1.10.csv' con 2894 URI univoci.


In [2]:
import pandas as pd
from rdflib import Graph

# 1. Definiamo in modo rigido la versione immutabile per il lineage dei dati
VERSION = "3.1.10" 
URL_VERSIONATO = f"http://qudt.org/{VERSION}/vocab/quantitykind"

print(f"Download della versione immutabile QUDT QuantityKind {VERSION}...")
g = Graph()
g.parse(URL_VERSIONATO, format="turtle")

# 2. Query di estrazione per le grandezze fisiche (QuantityKind)
query = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX qudt: <http://qudt.org/schema/qudt/>

SELECT ?quantityKindURI ?label ?description
WHERE {
  # Estraiamo le entità di tipo QuantityKind
  ?quantityKindURI rdf:type qudt:QuantityKind .
  
  # Estraiamo la label e filtriamo immediatamente per la lingua inglese
  ?quantityKindURI rdfs:label ?label .
  FILTER(lang(?label) = "en")
  
  # Opzionale: prendiamo anche la descrizione scientifica se presente
  OPTIONAL { ?quantityKindURI qudt:description ?description . }
}
"""

print("Esecuzione della query SPARQL locale...")
results = g.query(query)

# 3. Costruzione del dataset tabulare
rows = []
for row in results:
    rows.append({
        "QuantityKind_URI": str(row.quantityKindURI),
        "Label": str(row.label),
        "Description": str(row.description) if row.description else "",
        "Vocab_Version": f"QUDT-v{VERSION}"  # Tracciabilità della versione
    })

# 4. Generazione del file CSV finale
df = pd.DataFrame(rows)
output_file = f"qudt_quantitykinds_v{VERSION}.csv"
df.to_csv(output_file, index=False, encoding="utf-8")

print(f"Fatto! Generato il file '{output_file}' pronto per i tuoi modelli di metadati.")

Download della versione immutabile QUDT QuantityKind 3.1.10...
Esecuzione della query SPARQL locale...
Fatto! Generato il file 'qudt_quantitykinds_v3.1.10.csv' pronto per i tuoi modelli di metadati.


# BGS RockName

vai su: https://vocab.bgs.ac.uk/sparql

esegui:
'''
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?v WHERE { 
  ?s a skos:ConceptScheme . 
  ?s owl:versionInfo ?v . 
} LIMIT 1
'''

poi scarica il vocabolario da qui come .csv:
https://data.bgs.ac.uk/v/vocab/rthmtrlclss:RockName?_profile=prfl:dd&_mediatype=text/csv

e rinomina il file aggiungendo il primo numero dal risultato della query
2.5 - 2026-01 - Differentiated Addition & Original by definition changes 2.4 - 2026-01 - added scope notes to be used for use messages 2.3 - 2025-03 - minor errors in HTML fixed 2.2 - 2024-07 - fixed incorrect prefLabels for unstable & original in the RDF that used 'stable'. Also fixed preferred namespace URI. Updated Nicholas' affiliation to ANU 2.1 - 2023-05 - added colours 2.0 - 2023-01 - addition of Concepts addition & original, as per updated Registry Item status codes 1.2 - 2021-11 - changes IRI to reg-statuses; added unstable 1.1 - 2020-06 - altered structure & metadata, not content, to conform to Vocab Publication Profile 1.0 - 2018-06 - as per Registry Ontology original


# SWEET ontology for materials

In [3]:
import pandas as pd
from rdflib import Graph

# 1. Configurazione della versione e dei tre moduli nativi di SWEET
VERSION = "3.6.0"

sweet_modules = {
    "materials_industrial": {
        "url": "https://raw.githubusercontent.com/ESIPFed/sweet/master/src/matrIndustrial.ttl",
        "output_file": f"sweet_materials_industrial_v{VERSION}.csv",
        "source_tag": "SWEET_Industrial"
    },
    "physical_states": {
        "url": "https://raw.githubusercontent.com/ESIPFed/sweet/master/src/statePhysical.ttl",
        "output_file": f"sweet_physical_states_v{VERSION}.csv",
        "source_tag": "SWEET_PhysicalStates"
    }
}

# 2. Query SPARQL standard per isolare la sola lingua inglese ed eliminare duplicati
query = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?conceptURI ?label
WHERE {
  ?conceptURI rdfs:label ?label .
  FILTER(lang(?label) = "en")
}
ORDER BY UCASE(?label)
"""

print("Inizio della pipeline di estrazione modulare per SWEET...")
print("=" * 60)

# 3. Ciclo di parsing e scrittura indipendente
for module_key, config in sweet_modules.items():
    print(f"1. Download e parsing in RAM del file: {config['source_tag']}...")
    
    # Inizializziamo un grafo RDF fresco ad ogni ciclo per evitare contaminazioni di dati
    g = Graph()
    g.parse(config["url"], format="turtle")
    
    print(f"2. Esecuzione query semantica sul modulo...")
    results = g.query(query)
    
    rows = []
    for row in results:
        rows.append({
            "Concept_URI": str(row.conceptURI),
            "Label_EN": str(row.label),
            "Ontology_Source": config["source_tag"],
            "Vocab_Version": f"SWEET-v{VERSION}"  # Metadato di tracciabilità fisso
        })
        
    # Conversione in DataFrame e scrittura su disco
    df = pd.DataFrame(rows)
    df.to_csv(config["output_file"], index=False, encoding="utf-8")
    
    print(f"-> SUCCESSO: Generato '{config['output_file']}' con {len(df)} URI unici.")
    print("-" * 60)

print("=" * 60)
print("PROCESSO COMPLETATO! I due dizionari sono pronti e separati.")

Inizio della pipeline di estrazione modulare per SWEET...
1. Download e parsing in RAM del file: SWEET_Industrial...
2. Esecuzione query semantica sul modulo...
-> SUCCESSO: Generato 'sweet_materials_industrial_v3.6.0.csv' con 9 URI unici.
------------------------------------------------------------
1. Download e parsing in RAM del file: SWEET_PhysicalStates...
2. Esecuzione query semantica sul modulo...
-> SUCCESSO: Generato 'sweet_physical_states_v3.6.0.csv' con 83 URI unici.
------------------------------------------------------------
PROCESSO COMPLETATO! I due dizionari sono pronti e separati.


# EPOS msl vocabularies

Also reorganized

In [4]:
import json
import pandas as pd
import requests

# 1. Configurazione URL del file JSON Raw su GitHub (Versione 1.4)
VERSION = "1.4"
JSON_URL = "https://raw.githubusercontent.com/UtrechtUniversity/msl_vocabularies/main/vocabularies/combined/editor/1.4/editor_1-4.json"

print(f"1. Download del file JSON originale di MSL v{VERSION}...")
response = requests.get(JSON_URL, timeout=30)
response.raise_for_status()
json_data = response.json()

# Contenitore per tutte le righe piatte della tabella
flat_rows = []

# 2. Funzione ricorsiva per appiattire l'albero preservando cammini e chiavi annidate
def walk_and_flatten(node, current_path=[]):
    if isinstance(node, list):
        for item in node:
            walk_and_flatten(item, current_path)
            
    elif isinstance(node, dict):
        # Cerchiamo la label testuale controllando i sinonimi ammessi (priorità a 'text')
        label = ""
        for k in ["text", "label", "value", "name"]:
            if k in node and isinstance(node[k], str) and node[k].strip():
                label = node[k].strip()
                break
        
        if label:
            # Costruiamo il percorso gerarchico ad albero (Breadcrumb)
            new_path = current_path + [label]
            path_str = " -> ".join(new_path)
            
            # Estrazione sicura degli URI annidati nel campo 'extra'
            extra = node.get("extra", {})
            term_uri = ""
            vocab_uri = ""
            
            if isinstance(extra, dict):
                term_uri = extra.get("uri", "").strip()
                vocab_uri = extra.get("vocab_uri", "").strip()
            
            # Aggiungiamo il record piatto alla lista globale
            flat_rows.append({
                "Full_Path": path_str,
                "Label": label,
                "Concept_URI": term_uri,
                "Vocab_URI": vocab_uri,
                "Depth_Level": len(new_path) # Utile per capire il livello di nidificazione
            })
            
            # Se ci sono nodi figli ('children'), continuiamo a scendere ricorsivamente
            if "children" in node and node["children"]:
                walk_and_flatten(node["children"], new_path)
        else:
            # Gestione del dizionario radice o di macro-contenitori vuoti
            for v in node.values():
                if isinstance(v, (list, dict)):
                    walk_and_flatten(v, current_path)

print("2. Elaborazione dell'albero gerarchico e appiattimento dei nodi...")
walk_and_flatten(json_data)

# 3. Conversione in DataFrame ed esportazione in CSV piatto
df_flat = pd.DataFrame(flat_rows)

# Pulizia di sicurezza da record identici strutturali
df_flat.drop_duplicates(subset=["Full_Path", "Concept_URI"], inplace=True)

output_file = f"msl_combined_editor_v{VERSION}.csv"
df_flat.to_csv(output_file, index=False, encoding="utf-8")

print("=" * 70)
print(f"COMPLETATO! Generato il file piatto: '{output_file}'")
print(f"Totale concetti/nodi estratti complessivamente: {len(df_flat)}")
print("=" * 70)

1. Download del file JSON originale di MSL v1.4...
2. Elaborazione dell'albero gerarchico e appiattimento dei nodi...
COMPLETATO! Generato il file piatto: 'msl_combined_editor_v1.4.csv'
Totale concetti/nodi estratti complessivamente: 2994


## Reorganized

In [5]:
import csv
import os
import requests

# 1. Configurazione URL sorgente (MSL 1.4)
VERSION = "1.4"
JSON_URL = "https://raw.githubusercontent.com/UtrechtUniversity/msl_vocabularies/main/vocabularies/combined/editor/1.4/editor_1-4.json"

print(f"1. Download del file JSON di MSL v{VERSION}...")
response = requests.get(JSON_URL, timeout=30)
response.raise_for_status()
json_data = response.json()

# Contenitori globali per i dati e gli URI dei dizionari
dictionaries = {}
vocab_uris_per_code = {} 

# Regole di classificazione prioritarie basate sui tuoi tracciati CSV reali
def classify_path(path_str):
    p = path_str.lower()
    
    if "pore fluid" in p:
        return "MSL_POREFLUID", "Pore fluids"
    if "environment control" in p:
        return "MSL_ENVIRONMENTCONTROL", "Environment control"
    if "measured property" in p:
        return "MSL_MEASUREDPROPERTY", "Measured property"
    if "ancillary equipment" in p or "apparatus" in p or "equipment" in p:
        return "MSL_EQUIPMENT", "Ancillary equipment, Apparatus and Equipment"
    if "minerals" in p or "mineral" in p:
        return "MSL_MINERAL", "Mineral"
    if "analogue modelling material" in p or "synthesized material" in p:
        return "MSL_ARTIFICIAL", "Analogue modelling and synthesized materials"
    if any(r in p for r in ["sedimentary rock", "igneous rock", "metamorphic rock", "fault rock", "rock materials"]):
        return "MSL_ROCK", "Rock materials"
    if "unconsolidated sediment" in p or "sediment" in p:
        return "MSL_SEDIMENT", "Unconsolidated sediment materials"
    if "inferred" in p:
        return "MSL_INFERRED", "Inferred information"
    if "setting" in p:
        return "MSL_SETTING", "Setting"
    if "modeled" in p or "models" in p:
        return "MSL_MODEL", "Model"
    if "software" in p:
        return "MSL_SOFTWARE", "Software"
    if "analysis" in p or "technique" in p:
        return "MSL_ANALYSIS", "Analysis"
    if "facility names" in p or "facility name" in p:
        return "MSL_FACILITY", "Facility"
    if "facility types" in p or "facility type" in p:
        return "MSL_FACILITYTYPE", "Facility type"
    if "geological age" in p or "age" in p:
        return "MSL_GEOLOGICALAGE", "Geological age"
    if "analyzed feature" in p or "microstructure" in p:
        return "MSL_MICROSTRUCTURE", "Microstructure"
        
    return None, None

# 2. Funzione di scansione ricorsiva (Tree Walker)
def walk_and_route(node, current_path=[]):
    if isinstance(node, list):
        for item in node:
            walk_and_route(item, current_path)
            
    elif isinstance(node, dict):
        # Identifica dinamicamente la Label
        possible_label_keys = ["text", "label", "value", "name", "title", "prefLabel"]
        label = ""
        for k in possible_label_keys:
            if k in node and isinstance(node[k], str) and node[k].strip():
                label = node[k].strip()
                break
        
        if label:
            new_path = current_path + [label]
            path_str = " -> ".join(new_path)
            
            vocab_code, vocab_desc = classify_path(path_str)
            if vocab_code:
                if vocab_code not in dictionaries:
                    dictionaries[vocab_code] = {"description": vocab_desc, "items": []}
                if vocab_code not in vocab_uris_per_code:
                    vocab_uris_per_code[vocab_code] = set()
                
                extra = node.get("extra", {})
                if isinstance(extra, dict):
                    term_uri = extra.get("uri", "").strip()
                    vocab_uri = extra.get("vocab_uri", "").strip()
                    
                    if vocab_uri:
                        vocab_uris_per_code[vocab_code].add(vocab_uri)
                    
                    # Filtro per escludere i macro-contenitori di radice (quelli che finiscono in /)
                    is_container = term_uri.endswith('/') or term_uri == vocab_uri
                    
                    if term_uri and term_uri.startswith("http") and not is_container:
                        dictionaries[vocab_code]["items"].append({
                            "Concept_URI": term_uri,
                            "Label": label,
                            "Path_List": new_path,   # <--- SALVIAMO LA LISTA DEI LIVELLI PER IL SORTING
                            "Full_Path": path_str
                        })
            
            # Discende ricorsivamente nei rami figli
            possible_child_keys = ["children", "terms", "concepts", "items", "subconcepts"]
            children_found = False
            for ck in possible_child_keys:
                if ck in node and node[ck]:
                    walk_and_route(node[ck], new_path)
                    children_found = True
                    break
            
            if not children_found:
                for k, v in node.items():
                    if k not in possible_label_keys and k != "extra" and isinstance(v, (list, dict)):
                        walk_and_route(v, new_path)
        else:
            for v in node.values():
                if isinstance(v, (list, dict)):
                    walk_and_route(v, current_path)

print("2. Elaborazione gerarchica dell'albero e classificazione iniziale...")
walk_and_route(json_data)

print("\n3. Filtrazione, ordinamento dal 2° livello ed esportazione CSV...")
print("=" * 80)

os.makedirs("openbis_reference_vocabularies", exist_ok=True)

# 3. Scrittura dei file CSV con pulizia avanzata e sorting sul secondo livello
files_generated_count = 0
for vocab_code, data in dictionaries.items():
    filename = f"openbis_reference_vocabularies/{vocab_code}.csv"
    items = data["items"]
    
    all_uris = [item["Concept_URI"] for item in items]
    
    # A. FILTRAZIONE CONTENITORI INTERMEDI (es. esclude /apparatus se esiste /apparatus-deformation_testing)
    unique_items = []
    seen_uris = set()
    for item in items:
        uri = item["Concept_URI"]
        if uri in seen_uris:
            continue
            
        is_hyphen_container = any(other_uri.startswith(uri + "-") for other_uri in all_uris)
        if not is_hyphen_container:
            seen_uris.add(uri)
            unique_items.append(item)
            
    # B. SORTING FINALE IN BASE AL SECONDO LIVELLO (Risoluzione della richiesta)
    # Taglia il primo elemento della lista (index 0) e unisce il testo dal secondo livello in poi
    unique_items.sort(key=lambda x: " -> ".join(x["Path_List"][1:]).lower() if len(x["Path_List"]) > 1 else x["Full_Path"].lower())

    # Generazione stringa riga 3 (unione pipe)
    collected_vocab_uris = sorted(list(vocab_uris_per_code.get(vocab_code, set())))
    if collected_vocab_uris:
        global_vocabulary_description = "|".join(collected_vocab_uris)
    else:
        global_vocabulary_description = f"https://epos-msl.uu.nl/voc/{vocab_code.lower().split('_')[1]}/1.4/"

    with open(filename, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        
        # INTESTAZIONE OPENBIS (Righe 1-4)
        writer.writerow(["VOCABULARY_TYPE", "", ""])
        writer.writerow(["Code", "Description", ""])
        writer.writerow([vocab_code, global_vocabulary_description, ""])
        writer.writerow(["Code", "Description", "Label"])
        
        # SCRITTURA RECORD PURIFICATI E ORDINATI (Righe 5+)
        for idx, item in enumerate(unique_items, start=1):
            sequential_code = f"{vocab_code}{idx}"
            writer.writerow([sequential_code, item["Concept_URI"], item["Label"]])
            
    print(f"-> GENERATO: '{filename}' | Record atomici ordinati inseriti: {len(unique_items)}")
    files_generated_count += 1

print("=" * 80)
print(f"COMPLETATO! Pipeline conclusa. I CSV sono stati ordinati alfabeticamente a partire dal secondo livello del percorso.")

1. Download del file JSON di MSL v1.4...
2. Elaborazione gerarchica dell'albero e classificazione iniziale...

3. Filtrazione, ordinamento dal 2° livello ed esportazione CSV...
-> GENERATO: 'openbis_reference_vocabularies/MSL_ROCK.csv' | Record atomici ordinati inseriti: 180
-> GENERATO: 'openbis_reference_vocabularies/MSL_MINERAL.csv' | Record atomici ordinati inseriti: 298
-> GENERATO: 'openbis_reference_vocabularies/MSL_SEDIMENT.csv' | Record atomici ordinati inseriti: 24
-> GENERATO: 'openbis_reference_vocabularies/MSL_ARTIFICIAL.csv' | Record atomici ordinati inseriti: 90
-> GENERATO: 'openbis_reference_vocabularies/MSL_GEOLOGICALAGE.csv' | Record atomici ordinati inseriti: 121
-> GENERATO: 'openbis_reference_vocabularies/MSL_POREFLUID.csv' | Record atomici ordinati inseriti: 20
-> GENERATO: 'openbis_reference_vocabularies/MSL_SETTING.csv' | Record atomici ordinati inseriti: 171
-> GENERATO: 'openbis_reference_vocabularies/MSL_MODEL.csv' | Record atomici ordinati inseriti: 406
-> 

In [6]:
import csv
import os
import pandas as pd

# 1. Configurazione dei file di input (prodotti dalle tue celle precedenti) e dei relativi codici OpenBIS
VERSION_QUDT = "3.1.10"
VERSION_SWEET = "3.6.0"
VERSION_BGS = "2.5"

transformation_map = {
    "BGS_ROCKNAME": {
        "input_file": f"bgs_rock_name_v{VERSION_BGS}.csv",
        "global_uri": "https://vocab.bgs.ac.uk/v/vocab/rthmtrlclss:RockName",
        "col_uri": "iri",
        "col_label": "prefLabel"
    },"QUDT_UNIT": {
        "input_file": f"qudt_units_unique_v{VERSION_QUDT}.csv",
        "global_uri": "http://qudt.org/vocab/unit/",
        "col_uri": "Unit_URI",
        "col_label": "Label"
    },
    "QUDT_QUANTITYKIND": {
        "input_file": f"qudt_quantitykinds_v{VERSION_QUDT}.csv",
        "global_uri": "http://qudt.org/vocab/quantitykind/",
        "col_uri": "QuantityKind_URI",
        "col_label": "Label"
    },
    "SWEET_MATR_IND": {
        "input_file": f"sweet_materials_industrial_v{VERSION_SWEET}.csv",
        "global_uri": "http://sweetontology.net/matrIndustrial/",
        "col_uri": "Concept_URI",
        "col_label": "Label_EN"
    },
    "SWEET_STATE_PHYS": {
        "input_file": f"sweet_physical_states_v{VERSION_SWEET}.csv",
        "global_uri": "http://sweetontology.net/statePhysical/",
        "col_uri": "Concept_URI",
        "col_label": "Label_EN"
    }
}

print("Avvio della trasformazione strutturale in formato OpenBIS...")
print("=" * 80)

# Cartella di destinazione per tenere i file OpenBIS pronti e separati da quelli grezzi
output_dir = "openbis_reference_vocabularies"
os.makedirs(output_dir, exist_ok=True)

# 2. Ciclo di caricamento, conversione dei campi e scrittura dell'header a 4 righe
for vocab_code, cfg in transformation_map.items():
    if not os.path.exists(cfg["input_file"]):
        print(f"   [AVVISO] File '{cfg['input_file']}' non trovato. Salto la conversione di {vocab_code}.")
        print("-" * 80)
        continue
        
    print(f"1. Lettura del file grezzo: '{cfg['input_file']}'...")
    df_raw = pd.read_csv(cfg["input_file"])
    
    # Pulizia di sicurezza: eliminiamo eventuali righe dove l'URI o la Label sono nulli
    df_raw = df_raw.dropna(subset=[cfg["col_uri"], cfg["col_label"]])
    
    # Ordiniamo alfabeticamente per la Label per consistenza visiva nei menu di OpenBIS
    df_raw = df_raw.sort_values(by=cfg["col_label"], key=lambda col: col.str.lower())
    
    output_filename = f"{output_dir}/{vocab_code}.csv"
    
    with open(output_filename, mode="w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        
        # RIGHE 1-4: Generazione dell'intestazione metadati OpenBIS nativa
        writer.writerow(["VOCABULARY_TYPE", "", ""])
        writer.writerow(["Code", "Description", ""])
        writer.writerow([vocab_code, cfg["global_uri"], ""])  # L'URI dell'ontologia intera va qui
        writer.writerow(["Code", "Description", "Label"])
        
        # RIGHE 5+: Scrittura dei record atomici sequenziali con l'URI nella colonna Description
        for idx, (_, row) in enumerate(df_raw.iterrows(), start=1):
            sequential_code = f"{vocab_code}{idx}"
            uri_val = str(row[cfg["col_uri"]]).strip()
            label_val = str(row[cfg["col_label"]]).strip()
            
            writer.writerow([sequential_code, uri_val, label_val])
            
    print(f"-> TRASFORMATO: '{output_filename}' | Inseriti {len(df_raw)} termini strutturati.")
    print("-" * 80)

print("=" * 80)
print(f"PROCESSO COMPLETATO! I 5 dizionari OpenBIS con URI integrati sono pronti nella cartella '{output_dir}/'.")

Avvio della trasformazione strutturale in formato OpenBIS...
1. Lettura del file grezzo: 'bgs_rock_name_v2.5.csv'...
-> TRASFORMATO: 'openbis_reference_vocabularies/BGS_ROCKNAME.csv' | Inseriti 3113 termini strutturati.
--------------------------------------------------------------------------------
1. Lettura del file grezzo: 'qudt_units_unique_v3.1.10.csv'...
-> TRASFORMATO: 'openbis_reference_vocabularies/QUDT_UNIT.csv' | Inseriti 2894 termini strutturati.
--------------------------------------------------------------------------------
1. Lettura del file grezzo: 'qudt_quantitykinds_v3.1.10.csv'...
-> TRASFORMATO: 'openbis_reference_vocabularies/QUDT_QUANTITYKIND.csv' | Inseriti 1007 termini strutturati.
--------------------------------------------------------------------------------
1. Lettura del file grezzo: 'sweet_materials_industrial_v3.6.0.csv'...
-> TRASFORMATO: 'openbis_reference_vocabularies/SWEET_MATR_IND.csv' | Inseriti 9 termini strutturati.
-----------------------------